# ResNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，Residual Network（ResNet）を用いてCIFAR100データセットの100クラス物体認識を行い，スキップ構造（Residual Block）による深いネットワークの学習について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Residual Network（ResNet）
ResNetは，2015年のILSVRCで優勝したモデルであり，非常に深いネットワークを効率的に学習するための「スキップ構造」を導入したことで知られています．

ネットワークを深くすることで表現能力が向上し認識精度が改善する一方，あまりに深いネットワークでは勾配消失などにより効率的な学習が困難になります．ResNetは，通常のネットワークのようにある処理ブロックによる変換$F(x)$をそのまま次の層に渡すのではなく，スキップ構造によりブロックへの入力$x$をショートカットし，$H(x) = F(x) + x$を次の層に渡します．このショートカットを含めた処理単位をResidual Blockと呼びます．スキップ構造により，誤差逆伝播時に勾配が層をまたいで伝播できるため，非常に深いネットワークでも効率的な学習が可能になります．

Residual Blockは，3×3の畳み込み層とBatch Normalization，ReLUから構成されています．

### Basic BlockとBottleneck
Residual Blockには，BasicBlockとBottleneckの2種類の構造があります．BasicBlockは3×3の畳み込みを2つ用いた構造で，比較的浅いResNet（ResNet-18, 34など）に使用されます．一方，Bottleneckは1×1，3×3，1×1の3つの畳み込みを用いた構造で，一度チャンネル数を削減してから畳み込みを行い，再度元のチャンネル数に戻します．Bottleneck構造は深いResNet（ResNet-50, 101, 152など）に用いられ，同等の計算コストで精度を保ちながら効率化を図ることができます．

### CIFAR100への入力サイズの適応
ResNetには，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR10/100を対象とした「CIFAR版」の2種類の構造があります．両者はstem（最初の畳み込み）のカーネルサイズやチャンネル数，特徴マップサイズごとのブロック数などが異なります．本ノートブックでは，CIFAR100の入力サイズに合わせて，元論文でCIFAR実験に使用されたCIFAR版のResNet構造を実装します（両者の違いの詳細は本ノートブック末尾の「ImageNet版ResNetとCIFAR版ResNetの違い」を参照）．

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.convs(x)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, self.expansion * planes, kernel_size=1, bias=False),
            nn.BatchNorm2d(self.expansion * planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.convs(x)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

### ResNetの定義
上で定義したResidual Blockを用いてResNetを構築します．ここでは，使用するBlockの種類に応じて`ResNetBasicBlock`と`ResNetBottleneck`の2種類を定義します．

`depth`は構築するResNetの層数（例：20, 32, 44, 56, 110）を指定します．ResNetは，最初の畳み込み層と出力層（全結合層），および特徴マップサイズごとに3つに分かれたブロック内のResidual Blockから構成されるため，層数は

$$\text{depth} = (\text{Res. Block内の畳み込み数}) \times (\text{1ブロックあたりのRes. Block数}) \times 3 + 2$$

という関係になります．BasicBlockの場合は畳み込み数が2，Bottleneckの場合は3であるため，`depth`から1ブロックあたりのResidual Block数を逆算しています．

In [ ]:
class ResNetBasicBlock(nn.Module):
    def __init__(self, depth, n_class=100):
        super().__init__()
        # 指定した深さ（畳み込みの層数）でネットワークを構築できるかを確認
        assert (depth - 2) % 6 == 0, 'When use BasicBlock, depth should be 6n+2 (e.g. 20, 32, 44).'
        n_blocks = (depth - 2) // 6  # 1ブロックあたりのBasic Blockの数を決定

        self.inplanes = 16

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * BasicBlock.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * BasicBlock.expansion),
            )

        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * BasicBlock.expansion
        for _ in range(n_blocks - 1):
            layers.append(BasicBlock(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


class ResNetBottleneck(nn.Module):
    def __init__(self, depth, n_class=100):
        super().__init__()
        # 指定した深さ（畳み込みの層数）でネットワークを構築できるかを確認
        assert (depth - 2) % 9 == 0, 'When use Bottleneck, depth should be 9n+2 (e.g. 47, 56, 110).'
        n_blocks = (depth - 2) // 9  # 1ブロックあたりのBottleneckの数を決定

        self.inplanes = 16

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * Bottleneck.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * Bottleneck.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * Bottleneck.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * Bottleneck.expansion),
            )

        layers = [Bottleneck(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * Bottleneck.expansion
        for _ in range(n_blocks - 1):
            layers.append(Bottleneck(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`n_layers`でResNetの層数を指定し，`ResNetBasicBlock`または`ResNetBottleneck`を呼び出します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
# ResNetの層数を指定 (e.g. 20, 32, 44, 56, 110)
n_layers = 20

model = ResNetBasicBlock(depth=n_layers, n_class=100)    # BasicBlock構造を用いる場合
# model = ResNetBottleneck(depth=n_layers, n_class=100)  # Bottleneck構造を用いる場合
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版ResNetとCIFAR版ResNetの違い
このノートブックで実装したResNetは，元論文でCIFAR10/100の分類実験に使用された構造（CIFAR版）です．広く使われているImageNet版ResNetとは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 1層目の畳み込みのカーネルサイズ | 7×7 | 3×3 |
| 1層目の畳み込み後のチャンネル数 | 64 | 16 |
| 特徴マップサイズごとのブロック数 | 4 | 3 |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

ImageNet版のResNetは`torchvision.models`に実装されており，学習済みモデルも公開されています（[torchvisionのResNetリファレンス](https://pytorch.org/vision/stable/models.html#id10)）．

## 課題

1. ResNetの層数（`n_layers`）やBlockの種類（BasicBlock / Bottleneck）を変更して，パラメータ数や認識精度がどのように変化するか確認しましょう．

2. 学習の設定（ミニバッチサイズ，学習率，Epoch数，最適化手法など）を変更し，認識精度の変化を確認しましょう．

3. スキップ構造を取り除いた（`out += residual`を行わない）ネットワークを作成し，通常のネットワークとResNetとで学習の推移や認識精度がどのように異なるか比較しましょう．